In [1]:
import os
import re
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Paths
KT1_DIR = Path(r"D:\Lokesh files\Projects\KT1")
OUTPUT_DIR = Path(r"D:\Lokesh files\Projects\LearningDNA\processed\kt1_merged")

# Settings
FILES_PER_BATCH = 50
NUM_THREADS = 6

COLUMNS = ["timestamp", "solving_id", "question_id", "user_answer", "elapsed_time"]

SCHEMA = pa.schema([
    ("user_id", pa.int64()),
    ("timestamp", pa.int64()),
    ("solving_id", pa.int64()),
    ("question_id", pa.string()),
    ("user_answer", pa.string()),
    ("elapsed_time", pa.int64())
])

def get_user_id(path):
    match = re.fullmatch(r"u(\d+)", path.stem)
    return int(match.group(1)) if match else None

def process_batch(args):
    batch_id, files, output_dir = args
    output_file = Path(output_dir) / f"part_{batch_id:06d}.parquet"

    if output_file.exists():
        return batch_id, len(files), 0, 0, 0, []

    frames = []
    successful = 0
    failed = 0
    rows = 0
    errors = []

    for file in files:
        try:
            user_id = get_user_id(file)

            if user_id is None:
                errors.append((file.name, "Invalid filename"))
                continue

            df = pd.read_csv(file, low_memory=False)

            if list(df.columns) != COLUMNS:
                errors.append((file.name, "Unexpected columns"))
                continue

            df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce").astype("Int64")
            df["solving_id"] = pd.to_numeric(df["solving_id"], errors="coerce").astype("Int64")
            df["elapsed_time"] = pd.to_numeric(df["elapsed_time"], errors="coerce").astype("Int64")
            df["question_id"] = df["question_id"].astype("string")
            df["user_answer"] = df["user_answer"].astype("string")

            df.insert(0, "user_id", user_id)
            frames.append(df)

            successful += 1
            rows += len(df)

        except Exception as e:
            errors.append((file.name, str(e)))

    if frames:
        combined = pd.concat(frames, ignore_index=True)
        table = pa.Table.from_pandas(combined, schema=SCHEMA, preserve_index=False)
        pq.write_table(table, output_file, compression="snappy")

    return batch_id, len(files), successful, len(errors), rows, errors

if __name__ == "__main__":
    print("=" * 60)
    print("EDNET KT1 → PARQUET")
    print("=" * 60)
    print(f"Input   : {KT1_DIR}")
    print(f"Output  : {OUTPUT_DIR}")
    print(f"Threads : {NUM_THREADS}")
    print(f"Batch   : {FILES_PER_BATCH} files")
    print()

    if not KT1_DIR.exists():
        raise FileNotFoundError(f"KT1 folder not found: {KT1_DIR}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("Finding files...")
    start = time.time()

    user_files = list(KT1_DIR.glob("u*.csv"))
    user_files.sort(key=lambda x: get_user_id(x) or -1)

    print(f"Files found: {len(user_files):,}")
    print(f"Scan time: {time.time() - start:.1f}s")
    print()

    batches = [
        user_files[i:i + FILES_PER_BATCH]
        for i in range(0, len(user_files), FILES_PER_BATCH)
    ]

    print(f"Batches: {len(batches):,}")
    print()
    print("=" * 60)
    print("STARTING")
    print("=" * 60)

    tasks = [
        (i, batch, str(OUTPUT_DIR))
        for i, batch in enumerate(batches)
    ]

    start = time.time()
    completed = successful = failed = rows = files = 0

    with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
        futures = [executor.submit(process_batch, task) for task in tasks]

        for future in as_completed(futures):
            batch_id, batch_files, batch_success, batch_failed, batch_rows, errors = future.result()

            completed += 1
            files += batch_files
            successful += batch_success
            failed += batch_failed
            rows += batch_rows

            if errors:
                with open(OUTPUT_DIR / "failed_files.txt", "a", encoding="utf-8") as f:
                    for filename, error in errors:
                        f.write(f"{filename}\t{error}\n")

            elapsed = time.time() - start
            percent = completed / len(batches) * 100

            print(
                f"[{completed:,}/{len(batches):,}] "
                f"{percent:6.2f}% | "
                f"Files: {files:,} | "
                f"Rows: {rows:,} | "
                f"Failed: {failed:,} | "
                f"Time: {elapsed / 60:.1f} min"
            )

    parts = list(OUTPUT_DIR.glob("part_*.parquet"))
    size_gb = sum(p.stat().st_size for p in parts) / (1024 ** 3)

    print()
    print("=" * 60)
    print("COMPLETE")
    print("=" * 60)
    print(f"Files processed : {files:,}")
    print(f"Successful      : {successful:,}")
    print(f"Failed          : {failed:,}")
    print(f"Total rows      : {rows:,}")
    print(f"Parquet parts   : {len(parts):,}")
    print(f"Dataset size    : {size_gb:.2f} GB")
    print(f"Output          : {OUTPUT_DIR}")

    if parts:
        sample = pd.read_parquet(parts[0]).head(10)
        print()
        print("Sample:")
        print(sample)
        print()
        print("Data types:")
        print(sample.dtypes)

EDNET KT1 → PARQUET
Input   : D:\Lokesh files\Projects\KT1
Output  : D:\Lokesh files\Projects\LearningDNA\processed\kt1_merged
Threads : 6
Batch   : 50 files

Finding files...
Files found: 784,309
Scan time: 9.6s

Batches: 15,687

STARTING
[1/15,687]   0.01% | Files: 50 | Rows: 0 | Failed: 0 | Time: 0.0 min
[2/15,687]   0.01% | Files: 100 | Rows: 0 | Failed: 0 | Time: 0.0 min
[3/15,687]   0.02% | Files: 150 | Rows: 0 | Failed: 0 | Time: 0.0 min
[4/15,687]   0.03% | Files: 200 | Rows: 0 | Failed: 0 | Time: 0.0 min
[5/15,687]   0.03% | Files: 250 | Rows: 0 | Failed: 0 | Time: 0.0 min
[6/15,687]   0.04% | Files: 300 | Rows: 0 | Failed: 0 | Time: 0.0 min
[7/15,687]   0.04% | Files: 350 | Rows: 0 | Failed: 0 | Time: 0.0 min
[8/15,687]   0.05% | Files: 400 | Rows: 0 | Failed: 0 | Time: 0.0 min
[9/15,687]   0.06% | Files: 450 | Rows: 0 | Failed: 0 | Time: 0.0 min
[10/15,687]   0.06% | Files: 500 | Rows: 0 | Failed: 0 | Time: 0.0 min
[11/15,687]   0.07% | Files: 550 | Rows: 0 | Failed: 0 | Tim

In [2]:
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import pyarrow.parquet as pq

# Paths
DATASET = Path(r"D:\Lokesh files\Projects\LearningDNA\processed\kt1_merged")
NUM_THREADS = 6

EXPECTED_COLUMNS = [
    "user_id",
    "timestamp",
    "solving_id",
    "question_id",
    "user_answer",
    "elapsed_time"
]

VALID_ANSWERS = {"a", "b", "c", "d", "e"}

def validate_part(part):
    try:
        pf = pq.ParquetFile(part)
        rows = pf.metadata.num_rows
        schema = pq.read_schema(part)
        columns = schema.names

        if columns != EXPECTED_COLUMNS:
            return {
                "file": part.name,
                "rows": rows,
                "min_user": None,
                "max_user": None,
                "nulls": {},
                "negative_time": 0,
                "invalid_answers": 0,
                "duplicates": 0,
                "schema_error": True,
                "error": "Incorrect columns"
            }

        df = pd.read_parquet(part)

        nulls = df[EXPECTED_COLUMNS].isna().sum().to_dict()

        negative_time = int(
            (df["elapsed_time"] < 0).sum()
        )

        invalid_answers = int(
            (~df["user_answer"].isin(VALID_ANSWERS) &
             df["user_answer"].notna()).sum()
        )

        duplicates = int(
            df.duplicated().sum()
        )

        return {
            "file": part.name,
            "rows": len(df),
            "min_user": int(df["user_id"].min()) if len(df) else None,
            "max_user": int(df["user_id"].max()) if len(df) else None,
            "nulls": nulls,
            "negative_time": negative_time,
            "invalid_answers": invalid_answers,
            "duplicates": duplicates,
            "schema_error": False,
            "error": None
        }

    except Exception as e:
        return {
            "file": part.name,
            "rows": 0,
            "min_user": None,
            "max_user": None,
            "nulls": {},
            "negative_time": 0,
            "invalid_answers": 0,
            "duplicates": 0,
            "schema_error": True,
            "error": str(e)
        }

if __name__ == "__main__":
    print("=" * 60)
    print("KT1 MERGED DATASET VALIDATION")
    print("=" * 60)

    parts = sorted(DATASET.glob("part_*.parquet"))

    print(f"Parquet parts: {len(parts):,}")
    print(f"Threads: {NUM_THREADS}")
    print()

    if not parts:
        raise FileNotFoundError(
            f"No Parquet files found in {DATASET}"
        )

    start = time.time()

    total_rows = 0
    min_user = None
    max_user = None
    total_negative = 0
    total_invalid = 0
    total_duplicates = 0
    schema_errors = []
    failed_parts = []

    null_totals = {
        column: 0
        for column in EXPECTED_COLUMNS
    }

    completed = 0

    print("=" * 60)
    print("STARTING PARALLEL VALIDATION")
    print("=" * 60)

    with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
        futures = [
            executor.submit(validate_part, part)
            for part in parts
        ]

        for future in as_completed(futures):
            result = future.result()
            completed += 1

            if result["error"]:
                failed_parts.append(result)

            if result["schema_error"]:
                schema_errors.append(result)

            total_rows += result["rows"]
            total_negative += result["negative_time"]
            total_invalid += result["invalid_answers"]
            total_duplicates += result["duplicates"]

            if result["min_user"] is not None:
                min_user = (
                    result["min_user"]
                    if min_user is None
                    else min(min_user, result["min_user"])
                )

            if result["max_user"] is not None:
                max_user = (
                    result["max_user"]
                    if max_user is None
                    else max(max_user, result["max_user"])
                )

            for column, count in result["nulls"].items():
                null_totals[column] += count

            if completed % 100 == 0 or completed == len(parts):
                elapsed = time.time() - start
                percent = completed / len(parts) * 100

                print(
                    f"[{completed:,}/{len(parts):,}] "
                    f"{percent:6.2f}% | "
                    f"Rows: {total_rows:,} | "
                    f"Time: {elapsed / 60:.1f} min"
                )

    elapsed = time.time() - start

    print()
    print("=" * 60)
    print("VALIDATION RESULTS")
    print("=" * 60)

    print(f"Parquet parts       : {len(parts):,}")
    print(f"Total interactions  : {total_rows:,}")
    print(f"Minimum user_id     : {min_user}")
    print(f"Maximum user_id     : {max_user}")
    print(f"Schema errors       : {len(schema_errors):,}")
    print(f"Failed parts        : {len(failed_parts):,}")
    print(f"Negative elapsed    : {total_negative:,}")
    print(f"Invalid answers     : {total_invalid:,}")
    print(f"Duplicate rows      : {total_duplicates:,}")
    print(f"Validation time     : {elapsed / 60:.2f} min")

    print()
    print("Null values:")

    for column in EXPECTED_COLUMNS:
        print(
            f"  {column:15}: "
            f"{null_totals[column]:,}"
        )

    if schema_errors:
        print()
        print("Schema errors:")
        for result in schema_errors[:10]:
            print(
                f"  {result['file']}: "
                f"{result['error']}"
            )

    if failed_parts:
        print()
        print("Failed parts:")
        for result in failed_parts[:10]:
            print(
                f"  {result['file']}: "
                f"{result['error']}"
            )

    print()
    print("=" * 60)

    if not schema_errors and not failed_parts:
        print("PASS: All Parquet parts are readable.")
    else:
        print("WARNING: Some parts need investigation.")

    print("=" * 60)

KT1 MERGED DATASET VALIDATION
Parquet parts: 15,687
Threads: 6

STARTING PARALLEL VALIDATION
[100/15,687]   0.64% | Rows: 6,218,919 | Time: 0.2 min
[200/15,687]   1.27% | Rows: 12,124,427 | Time: 0.4 min
[300/15,687]   1.91% | Rows: 16,476,065 | Time: 0.6 min
[400/15,687]   2.55% | Rows: 18,813,758 | Time: 0.7 min
[500/15,687]   3.19% | Rows: 21,455,968 | Time: 0.8 min
[600/15,687]   3.82% | Rows: 23,076,225 | Time: 0.9 min
[700/15,687]   4.46% | Rows: 24,394,349 | Time: 0.9 min
[800/15,687]   5.10% | Rows: 25,717,324 | Time: 1.0 min
[900/15,687]   5.74% | Rows: 26,993,855 | Time: 1.1 min
[1,000/15,687]   6.37% | Rows: 27,785,205 | Time: 1.1 min
[1,100/15,687]   7.01% | Rows: 28,833,043 | Time: 1.2 min
[1,200/15,687]   7.65% | Rows: 29,688,026 | Time: 1.3 min
[1,300/15,687]   8.29% | Rows: 30,390,191 | Time: 1.3 min
[1,400/15,687]   8.92% | Rows: 31,021,359 | Time: 1.4 min
[1,500/15,687]   9.56% | Rows: 31,805,157 | Time: 1.4 min
[1,600/15,687]  10.20% | Rows: 32,677,364 | Time: 1.5 mi